In [2]:
from experiences.attention_only_transformer import load_model

model_, model = load_model()

In [3]:
from zonotope.zonotope import Zonotope
import torch as t
from einops import einsum

In [4]:
prompt = "That's me in the corner, that's me in the spot, losing my religion. That's me in the corner, that's me in the"

str_tokens = model.to_str_tokens(prompt)
print(str_tokens)

tokens = model.to_tokens(prompt)
tokens.shape

['<|endoftext|>', 'That', "'s", ' me', ' in', ' the', ' corner', ',', ' that', "'s", ' me', ' in', ' the', ' spot', ',', ' losing', ' my', ' religion', '.', ' That', "'s", ' me', ' in', ' the', ' corner', ',', ' that', "'s", ' me', ' in', ' the']


torch.Size([1, 31])

In [11]:
resid_pre = model_.embed(tokens)
shortformer_pos_embed = model_.pos_embed(tokens)
attn = model_.blocks[0].attn

q = (
    einsum(
        resid_pre + shortformer_pos_embed,
        attn.W_Q,
        "batch posn d_model, nheads d_model d_head -> batch posn nheads d_head",
    )
    + attn.b_Q
)
k = (
    einsum(
        resid_pre + shortformer_pos_embed,
        attn.W_K,
        "batch posn d_model, nheads d_model d_head -> batch posn nheads d_head",
    )
    + attn.b_K
)
v = (
    einsum(
        resid_pre,
        attn.W_V,
        "batch posn d_model, nheads d_model d_head -> batch posn nheads d_head",
    )
    + attn.b_V
)
q.shape, k.shape, v.shape

(torch.Size([1, 31, 12, 64]),
 torch.Size([1, 31, 12, 64]),
 torch.Size([1, 31, 12, 64]))

In [12]:
z = Zonotope.from_values(resid_pre)
z.shape

/home/sckathach/desktop/zonotopes/zonotope/zonotope.py:87: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  center = t.tensor(center_values, dtype=t.float)


torch.Size([1, 31, 768])

In [15]:
values = (
    z.mul(
        attn.W_V,
        "batch seq d_model, n_heads d_model d_head -> batch seq n_heads d_head",
    )
    + attn.b_V
)

z_with_pos = z + shortformer_pos_embed

queries = (
    z_with_pos.mul(
        attn.W_Q,
        "batch seq d_model, n_heads d_model d_head -> batch seq n_heads d_head",
    )
    + attn.b_Q
)
keys = (
    z_with_pos.mul(
        attn.W_K,
        "batch seq d_model, n_heads d_model d_head -> batch seq n_heads d_head",
    )
    + attn.b_K
)
queries.shape, keys.shape, values.shape


(torch.Size([1, 31, 12, 64]),
 torch.Size([1, 31, 12, 64]),
 torch.Size([1, 31, 12, 64]))

In [17]:
from zonotope.functional import dot_product

attn_scores = einsum(
    q,
    k,
    "batch posn_Q nheads d_head, batch posn_K nheads d_head -> batch nheads posn_Q posn_K",
)
attn_scores_z = dot_product(
    queries,
    keys,
    "batch posn_Q nheads d_head, batch posn_K nheads d_head -> batch nheads posn_Q posn_K",
)

RuntimeError: Sizes of tensors must match except in dimension 4. Expected size 31 but got size 12 for tensor number 1 in the list.

In [16]:
t.allclose(queries.W_C, q), t.allclose(values.W_C, v), t.allclose(keys.W_C, k)

(True, True, True)

In [5]:
z = Zonotope(center=flat_embeds)
D = model.cfg.d_model
S = N // D
model.W_Q.shape

torch.Size([2, 12, 768, 64])

In [6]:
queries_center = einsum(
    model.W_Q[0],
    z.W_C.view(S, D),
    "n_heads d_model d_heads, seq_len d_model -> seq_len n_heads d_heads",
)
queries_center.shape

torch.Size([31, 12, 64])

In [7]:
flat = queries_center.reshape(-1)
oui = queries_center[2]
shap = oui.shape

flat.shape, shap

(torch.Size([23808]), torch.Size([12, 64]))

In [8]:
flat.reshape(31, *shap).shape

torch.Size([31, 12, 64])

In [9]:
queries_center.reshape(-1).shape

torch.Size([23808])

In [2]:
from tests.test_zonotopes import create_zonotope

zz = create_zonotope([1, 2, 3], infinity_terms=t.eye(3))
a = t.Tensor([1, 2, 3]).float()

da = einsum(a, a, "N, N ->")
da

/home/sckathach/desktop/zonotopes/zonotope/utils.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  infinity_terms = t.tensor(infinity_terms, dtype=t.float16)


tensor(14.)

In [3]:
from zonotope.functional import dot_product

dr = dot_product(zz, zz)
dr.W_C

tensor(15.5000, dtype=torch.float16)

In [4]:
dr.W_Ei

tensor([[2.0000, 4.0000, 6.0000, 1.5000]], dtype=torch.float16)